# Fashion MNIST Multimodales Modell

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_11/04_fashion_mnist_multi_input.ipynb)

Dieses Notebook implementiert ein multimodales Modell für den Fashion MNIST Datensatz. Das Modell kombiniert Bilddaten (CNN) und Textdaten (LSTM), um Kleidungsstücke zu klassifizieren.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")

## 1. Daten laden und vorbereiten

Wir laden den Fashion MNIST Datensatz und erstellen für jedes Bild einen entsprechenden Satz: "this is a [Klassenname]".

In [ ]:
# Fashion MNIST-Daten laden
fashion_mnist = keras.datasets.fashion_mnist
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()

# Normalisierung der Bilddaten auf den Bereich [0, 1]
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Kanal-Dimension hinzufügen (für CNN notwendig)
x_train_full = np.expand_dims(x_train_full, -1)
x_test = np.expand_dims(x_test, -1)

# Aufteilung in Trainings- und Validierungsdaten
x_train, x_valid = x_train_full[:-5000], x_train_full[-5000:]
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

class_names = ["t-shirt/top", "trouser", "pullover", "dress", "coat",
               "sandal", "shirt", "sneaker", "bag", "ankle boot"]

def generate_sentences(labels):
    return [f"this is a {class_names[label]}" for label in labels]

train_sentences = generate_sentences(y_train)
valid_sentences = generate_sentences(y_valid)
test_sentences = generate_sentences(y_test)

print(f"Beispiel-Satz: {train_sentences[0]}")

### Text-Vektorisierung

Die Sätze müssen in ein numerisches Format umgewandelt werden, damit das LSTM sie verarbeiten kann.

In [ ]:
tokenizer = keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(train_sentences)

train_seq = tokenizer.texts_to_sequences(train_sentences)
valid_seq = tokenizer.texts_to_sequences(valid_sentences)
test_seq = tokenizer.texts_to_sequences(test_sentences)

# Padding auf die gleiche Länge
max_len = max(len(s) for s in train_seq)
train_seq = keras.preprocessing.sequence.pad_sequences(train_seq, maxlen=max_len)
valid_seq = keras.preprocessing.sequence.pad_sequences(valid_seq, maxlen=max_len)
test_seq = keras.preprocessing.sequence.pad_sequences(test_seq, maxlen=max_len)

vocab_size = len(tokenizer.word_index) + 1
print(f"Vokabulargröße: {vocab_size}")
print(f"Maximale Sequenzlänge: {max_len}")

## 2. Modellarchitektur

Wir nutzen die Functional API von Keras, um ein Modell mit zwei Eingängen zu erstellen:
1. Ein **CNN** für die Bildverarbeitung.
2. Ein **LSTM** für die Textverarbeitung.

Die Merkmale beider Zweige werden anschließend kombiniert.

In [ ]:
# --- Bild-Zweig (CNN) ---
image_input = layers.Input(shape=(28, 28, 1), name="image_input")
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(image_input)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Flatten()(x)
image_features = layers.Dense(64, activation="relu")(x) # 64-dim wie in der Grafik

# --- Text-Zweig (LSTM) ---
text_input = layers.Input(shape=(max_len,), name="text_input")
y = layers.Embedding(input_dim=vocab_size, output_dim=16)(text_input)
y = layers.LSTM(48)(y) # 48-dim wie in der Grafik
text_features = y

# --- Fusion und gemeinsames Head ---
concat = layers.Concatenate()([image_features, text_features])
z = layers.Dense(256, activation="relu")(concat) # 256-dim wie in der Grafik
z = layers.Dense(128, activation="relu")(z) # 128-dim wie in der Grafik
output = layers.Dense(10, activation="softmax")(z)

model = keras.Model(inputs=[image_input, text_input], outputs=output)

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

## 3. Training

Das Modell wird über 10 Epochen trainiert.

In [ ]:
history = model.fit(
    [x_train, train_seq], y_train,
    epochs=10,
    validation_data=([x_valid, valid_seq], y_valid),
    batch_size=32
)

## 4. Visualisierung des Trainingsverlaufs

Plot des Trainings- und Validierungsverlusts über die Epochen.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Trainingsverlust')
plt.plot(history.history['val_loss'], label='Validierungsverlust')
plt.title('Trainings- und Validierungsverlust')
plt.xlabel('Epochen')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 5. Ergebnisse auf dem Testdatensatz

Abschließende Bewertung des Modells mit den Testdaten.

In [ ]:
test_loss, test_acc = model.evaluate([x_test, test_seq], y_test, verbose=0)
print(f"Test-Loss: {test_loss:.4f}")
print(f"Test-Genauigkeit: {test_acc:.4f}")